In [ ]:
#| default_exp basic

# Basic
> Fundamental Python utilities for data manipulation, type checking, and common operations

This module provides battle-tested helpers for working with nested data structures, dictionaries, strings, and objects. Most functions here solve recurring problems that appear across projects.

**Key capabilities:**
- Access nested data with dot notation (`val_at`, `vals_at`)
- Enhanced dict with attribute access (`AD`)
- Dictionary manipulation helpers (`pops_`, `gets`, `update_`)
- String utilities (`shorten`)
- Type checking (`is_listy`, `empty`)
- Function composition (`Runner`)
- Session ID generation (`id_gen`)

**When to use this module:** When you need practical utilities for data manipulation without bringing in heavy dependencies.

In [ ]:
import importlib
import inspect
import operator
import os
import pprint
import re
import sys
from binascii import hexlify
from collections import defaultdict
from inspect import Parameter
from pathlib import Path
from types import ModuleType
from typing import Any
from typing import Callable
from typing import Hashable
from typing import Iterable
from typing import Literal
from typing import Mapping
from typing import MutableMapping
from typing import Self
from typing import Sequence
from typing import TypeAlias
from typing import TypeVar

import fastcore.all as FC

In [ ]:
#| hide
import dataclasses
import json
import random
import re
import time
from collections import deque
from datetime import datetime
from functools import reduce
from IPython.display import Markdown
from types import NoneType
from types import SimpleNamespace
from typing import Type

from fastcore.test import *
from fastcore.xtras import dict2obj
from fastcore.xdg import xdg_data_home

In [ ]:
#| hide
from pote.test import *

# timing

This dialog is taking a very long time to `Run All`. Write a function to get the time each message took to run returning a map of message id to duration.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The stored `time` values are completion timestamps, so subtracting consecutive timed messages gives a useful `Run All` profile. The first timed message has no earlier timestamp, so its duration is unknown.

```python
#| export
def msg_run_durations(path):
    "Return approximate Run All durations in seconds, keyed by message id."
    cells = json.loads(Path(path).read_text())['cells']
    times = [(o['id'], datetime.fromisoformat(o['metadata']['time'])) for o in cells if o.get('metadata', {}).get('time')]
    return dict((id, None if i == 0 else (t-times[i-1][1]).total_seconds()) for i,(id,t) in enumerate(times))
```

```python
tmp = Path('/tmp/msg_times.ipynb')
tmp.write_text(json.dumps(dict(cells=[
    dict(id='a', metadata=dict(time='2026-08-25T12:00:00+02:00')),
    dict(id='b', metadata=dict(time='2026-08-25T12:00:00.250000+02:00'))
])))
test_eq(msg_run_durations(tmp), dict(a=None, b=.25))
tmp.unlink()
```

This is meaningful for a contiguous `Run All`; separately executed messages will include any idle time between runs.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 65045, "completion_tokens": 4942, "total_tokens": 69987, "cached_tokens": 3456, "reasoning_tokens": 4660, "cost": 0.01346325}
```


Use `find_msgs`, no need to read the .ipynb file.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
from dialoghelper.solveitskill import find_msgs
```

```python
#| export
async def msg_run_durations(dname=''):
    "Return approximate Run All durations in seconds, keyed by message id."
    msgs = await find_msgs(dname=dname)
    times = [(m['id'], datetime.fromisoformat(m['time_run'])) for m in msgs if m['time_run']]
    return dict((id, None if i == 0 else (t-times[i-1][1]).total_seconds()) for i,(id,t) in enumerate(times))
```

```python
await msg_run_durations()
```

The first timed message remains `None`: there is no recorded preceding timestamp from which to infer its duration.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 65398, "completion_tokens": 676, "total_tokens": 66074, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.0103167}
```


In [ ]:
# async def msg_run_durations(dname=''):
#     "Return approximate Run All durations in seconds, keyed by message id."
#     msgs = await find_msgs(dname=dname)
#     times = [(m['id'], datetime.fromisoformat(m['time_run'])) for m in msgs if m['time_run']]
#     return dict((id, None if i == 0 else (t-times[i-1][1]).total_seconds()) for i,(id,t) in enumerate(times))

In [ ]:
# ms = sorted((await msg_run_durations()).items(), key=lambda x: x[1] or 0, reverse=True)
# Markdown('\n\n'.join([f"#_{mid}: {t}" for mid,t in ms]))

What about of using IPython hooks to maintain a global mapping of message-id to time?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

- ⏳ `web_search(type="search", queries="['site:ipython.readthedocs.io pre_run_ce…", query="site:ipython.readthedocs.io pre_run_cell…")` ⏳


```json {.tool}
{
  "id": "ws_0ca58e2be1f881b1016a8d753174c887d1836f01c2b8b33f89",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:ipython.readthedocs.io pre_run_cell post_run_cell ExecutionInfo cell_id",
      "site:github.com/ipython/ipython ExecutionInfo cell_id pre_run_cell"
    ],
    "query": "site:ipython.readthedocs.io pre_run_cell post_run_cell ExecutionInfo cell_id"
  },
  "result": "Server tool call executed.",
  "server": true
}
```





## Hook-based timing

Yes. This measures each execution directly, rather than inferring durations from persisted completion timestamps. IPython’s `pre_run_cell` and `post_run_cell` events expose the frontend-provided `cell_id`, which Solveit can use as the message id. ([ipython.readthedocs.io](https://ipython.readthedocs.io/en/9.16.0/config/callbacks.html?utm_source=openai))
[*](https://ipython.readthedocs.io/en/9.16.0/config/callbacks.html?utm_source=openai "IPython Events — IPython 9.16.0 documentation")
```python
from contextlib import suppress
from time import perf_counter
```

```python
#| export
def install_msg_timer():
    "Install hooks that record interactive execution seconds by message id."
    ip,old = get_ipython(),globals().get('_msg_timer')
    if old:
        for event,name in dict(pre_run_cell='pre', post_run_cell='post').items():
            with suppress(ValueError): ip.events.unregister(event, old[name])
    starts,times = dict(),dict()
    def pre(info):
        if (id := info.cell_id) is not None: starts[id] = perf_counter()
    def post(res):
        if (id := res.info.cell_id) in starts: times[id] = perf_counter()-starts.pop(id)
    state = dict(pre=pre, post=post, times=times)
    ip.events.register('pre_run_cell', pre)
    ip.events.register('post_run_cell', post)
    globals()['_msg_timer'] = state
    return times

install_msg_timer()
```

The installation cell itself cannot be timed because its pre-execution hook did not yet exist. This following cell establishes a first recorded entry:

```python
sum(range(1_000_000))
```

```python
_msg_timer['times']
```

Re-running `install_msg_timer()` replaces the old hooks, avoiding duplicate measurements.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 47069, "completion_tokens": 3582, "total_tokens": 50651, "cached_tokens": 3456, "reasoning_tokens": 3213, "cost": 0.00974685}
```


perhaps `install_msg_timer` should return a cleanup function.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes. Return `cleanup`, which unregisters the hooks and returns the collected timings.

```python
#| export
def install_msg_timer():
    "Install message timing hooks; return a function that removes them."
    ip,old = get_ipython(),globals().get('_msg_timer')
    if old: old['cleanup']()
    starts,times = {},{}
    def pre(info):
        if (id := info.cell_id) is not None: starts[id] = perf_counter()
    def post(res):
        if (id := res.info.cell_id) in starts: times[id] = perf_counter()-starts.pop(id)
    def cleanup():
        for event,fn in dict(pre_run_cell=pre, post_run_cell=post).items():
            with suppress(ValueError): ip.events.unregister(event, fn)
        return times
    ip.events.register('pre_run_cell', pre)
    ip.events.register('post_run_cell', post)
    globals()['_msg_timer'] = dict(cleanup=cleanup, times=times)
    return cleanup
```

```python
cleanup = install_msg_timer()
callable(cleanup), cleanup()
```

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 40291, "completion_tokens": 695, "total_tokens": 40986, "cached_tokens": 3456, "reasoning_tokens": 455, "cost": 0.006564899999999999}
```


In [ ]:
# from contextlib import suppress
# from time import perf_counter

In [ ]:
# def install_msg_timer():
#     "Install message timing hooks; return a function that removes them."
#     ip,old = get_ipython(),globals().get('_msg_timer')
#     if old: old['cleanup']()
#     starts,times = {},{}
#     def pre(info):
#         if (id := info.cell_id) is not None: starts[id] = perf_counter()
#     def post(res):
#         if (id := res.info.cell_id) in starts: times[id] = perf_counter()-starts.pop(id)
#     def cleanup():
#         for event,fn in dict(pre_run_cell=pre, post_run_cell=post).items():
#             with suppress(ValueError): ip.events.unregister(event, fn)
#         return times
#     ip.events.register('pre_run_cell', pre)
#     ip.events.register('post_run_cell', post)
#     globals()['_msg_timer'] = dict(cleanup=cleanup, times=times)
#     return cleanup

In [ ]:
# cleanup = install_msg_timer()

In [ ]:
# _msg_timer['times']

In [ ]:
# sum(range(1_000_000))

In [ ]:
# _msg_timer['times']

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`_msg_timer['times']` stayed empty because this kernel is not populating `ExecutionInfo.cell_id`; the hooks run, but the guard rejects every execution.

Inspect the hook payload and kernel request metadata before choosing the message-id source:

```python
_hook_info = {}

def inspect_post(res):
    _hook_info.update(info=vars(res.info), parent=get_ipython().parent_header)

get_ipython().events.register('post_run_cell', inspect_post)
```

```python
1
```

```python
_hook_info
```

This will show whether Solveit supplies its message id under another field or in `parent_header` metadata.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 36500, "completion_tokens": 654, "total_tokens": 37154, "cached_tokens": 8576, "reasoning_tokens": 516, "cost": 0.0059655}
```


In [ ]:
# _hook_info = {}
# 
# def inspect_post(res):
#     _hook_info.update(info=vars(res.info), parent=get_ipython().parent_header)
# 
# get_ipython().events.register('post_run_cell', inspect_post)

In [ ]:
# 1

In [ ]:
# _hook_info

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`ExecutionInfo` is slotted, so `vars(res.info)` fails before `_hook_info` is updated. Capture its relevant fields explicitly instead.

```python
with suppress(ValueError): get_ipython().events.unregister('post_run_cell', inspect_post)

_hook_info = {}
def inspect_post(res):
    i = res.info
    _hook_info.update(execution_count=res.execution_count, cell_id=i.cell_id, cell_meta=i.cell_meta,
                      parent_header=dict(get_ipython().parent_header))

get_ipython().events.register('post_run_cell', inspect_post)
```

```python
1
```

```python
_hook_info
```

This should show the request header and confirm whether any Solveit-specific message identifier is available there.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 37172, "completion_tokens": 674, "total_tokens": 37846, "cached_tokens": 3456, "reasoning_tokens": 516, "cost": 0.0060812999999999996}
```


In [ ]:
# with suppress(ValueError): get_ipython().events.unregister('post_run_cell', inspect_post)
# 
# _hook_info = {}
# def inspect_post(res):
#     i = res.info
#     _hook_info.update(execution_count=res.execution_count, cell_id=i.cell_id, cell_meta=i.cell_meta,
#                       parent_header=dict(get_ipython().parent_header))
# 
# get_ipython().events.register('post_run_cell', inspect_post)

In [ ]:
# 1

In [ ]:
# _hook_info

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The callback is still failing before it reaches `_hook_info.update()`. The likely failing expression is `get_ipython().parent_header`: the parent header belongs to the kernel, not the interactive shell.

Replace the callback with this narrower, defensive version:

```python
ip = get_ipython()
with suppress(ValueError): ip.events.unregister('post_run_cell', inspect_post)

_hook_info = {}
def inspect_post(res):
    i = res.info
    hdr = getattr(getattr(ip, 'kernel', None), '_parent_header', {})
    _hook_info.update(execution_count=res.execution_count, cell_id=i.cell_id, cell_meta=i.cell_meta, parent_header=hdr)

ip.events.register('post_run_cell', inspect_post)
```

Then execute one cell and inspect the capture:

```python
1
```

```python
_hook_info
```

This will establish whether the kernel request header contains a Solveit message identifier.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 37903, "completion_tokens": 900, "total_tokens": 38803, "cached_tokens": 36224, "reasoning_tokens": 705, "cost": 0.006360450000000001}
```


In [ ]:
# ip = get_ipython()
# with suppress(ValueError): ip.events.unregister('post_run_cell', inspect_post)
# 
# _hook_info = {}
# def inspect_post(res):
#     i = res.info
#     hdr = getattr(getattr(ip, 'kernel', None), '_parent_header', {})
#     _hook_info.update(res=res, execution_count=res.execution_count, cell_id=i.cell_id, cell_meta=i.cell_meta, parent_header=hdr)
# 
# ip.events.register('post_run_cell', inspect_post)

In [ ]:
# 1

In [ ]:
# _hook_info

In [ ]:
# dir(_hook_info['res'])

In [ ]:
# ip = get_ipython()
# with suppress(ValueError): ip.events.unregister('post_run_cell', inspect_post)

And what does the pre hook receive?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`pre_run_cell` receives a single `ExecutionInfo` object. Capture successive calls so inspecting the result does not overwrite the observation:

```python
_pre_events = []

def inspect_pre(info):
    _pre_events.append(dict(raw_cell=info.raw_cell, transformed_cell=info.transformed_cell, store_history=info.store_history,
                            silent=info.silent, shell_futures=info.shell_futures, cell_id=info.cell_id, cell_meta=info.cell_meta))

ip.events.register('pre_run_cell', inspect_pre)
```

```python
1
```

```python
_pre_events
```

The entry for `1` will show exactly what the pre-hook receives. It will likely confirm that `cell_id` and `cell_meta` are also `None`, so the IPython event system alone cannot map executions back to Solveit message IDs.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 38716, "completion_tokens": 698, "total_tokens": 39414, "cached_tokens": 36224, "reasoning_tokens": 516, "cost": 0.0063309}
```


In [ ]:
# _pre_events = []
# 
# def inspect_pre(info):
#     _pre_events.append(dict(raw_cell=info.raw_cell, transformed_cell=info.transformed_cell, store_history=info.store_history,
#                             silent=info.silent, shell_futures=info.shell_futures, cell_id=info.cell_id, cell_meta=info.cell_meta))
# 
# ip.events.register('pre_run_cell', inspect_pre)

In [ ]:
# 1

In [ ]:
# _pre_events

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The hook can time executions, but this kernel supplies neither `cell_id` nor request metadata:

```python
{'execution_count': 29, 'cell_id': None, 'cell_meta': None, 'parent_header': {}}
```

So hooks alone cannot produce a reliable `{message_id: duration}` map. They can only produce `{execution_count: duration}`.

The clean fix is for Solveit to send the message id as `cell_id` with each `execute_request`; then the original timer works unchanged. Without that frontend/kernel change, mapping an execution back to a dialog message would require an unreliable heuristic such as matching source text.

```json {.usage}
{"model": "gpt-5.6-terra", "prompt_tokens": 38363, "completion_tokens": 652, "total_tokens": 39015, "cached_tokens": 37248, "reasoning_tokens": 516, "cost": 0.006243450000000001}
```


That's weird.

In [ ]:
# import ipykernel_helper
# dir(ipykernel_helper)

In [ ]:
# await add_msg(symsrc('ipykernel_helper.core'), msg_type='raw')

In [ ]:
# import solveit

In [ ]:
# dir(solveit)

In [ ]:
# solveit.__spec__

In [ ]:
# cleanup()

# Type Utilities

## empty

A sentinel value for distinguishing "no value provided" from `None`. Useful in function signatures where `None` is a valid argument.

In [ ]:
#| exporti

class Empty(type): __repr__ = __str__ = lambda self: 'empty'
class EmptyT(metaclass=Empty):...

# _EMPTY: TypeAlias = Parameter.empty
# EmptyT = Type[_EMPTY]

In [ ]:
empty = EmptyT

def is_empty(x) -> bool: return x is empty

In [ ]:
test_is(str(empty), 'empty')
empty

empty

In [ ]:
def test_empty() -> Empty: return empty
test_eq_type(test_empty(), empty)

# Data Structures

## AD

Attribute Dict - a dictionary that also allows attribute-style access. Perfect for configuration objects or API responses where you want both `config['key']` and `config.key` syntax.

In [ ]:
_VT = TypeVar('_VT')
# from `fastcore` + generics
class AD(dict[str, _VT]):
    "`dict` subclass that also provides access to keys as attrs"
    def __getattr__(self, k:str) -> _VT: return self[k] if k in self else FC.stop(AttributeError(k))  # type: ignore
    def __setattr__(self, k, v:_VT): (self.__setitem__, super().__setattr__)[k[0]=='_'](k,v)
    def __dir__(self) -> Iterable[str]: return super().__dir__() + list(self.keys())  # type: ignore
    def _repr_markdown_(self): return f'```json\n{pprint.pformat(self, indent=2)}\n```'
    def copy(self) -> Self: return type(self)(**self)

In [ ]:
# Basic usage: dict and attribute access
ad = AD(a=1, b=2)
test_eq(ad.a, 1)
test_eq(ad['a'], 1)

ad.b = 3
test_eq(ad.b, 3)
ad.update(a=4, b=5)
test_eq(ad.b, 5)

In [ ]:
# Edge cases: empty dict, missing keys
ad = AD()
test_fail(lambda: ad.a)
test_fail(lambda: ad['b'])


## is_listy

Test whether `x` is iterable but not a string or bytes. Useful for duck-typing checks where you want to handle collections but not strings.

In [ ]:
def is_listy(x, exclude=None):
    "Return True if x is iterable but not a string or bytes, with optional type exclusions."
    return isinstance(x, Iterable) and not isinstance(x, (bytes, str)) and not bool(exclude and isinstance(x, exclude))

def is_listy_type(x, exclude=None):
    "Return True if x is an iterable type but not str or bytes, with optional type exclusions."
    return issubclass(x, Iterable) and not issubclass(x, (bytes, str)) and not bool(exclude and issubclass(x, exclude))

In [ ]:
# Common usage: distinguish collections from strings
test_is(is_listy([1, 2, 3]), True)
test_is(is_listy((1, 2, 3)), True)
test_is(is_listy({'a': 1}), True)
test_is(is_listy('hello'), False)  # strings are NOT listy
test_is(is_listy(b'bytes'), False)  # bytes are NOT listy
test_is(is_listy(None), False)

In [ ]:
# Coverage: ranges, generators, and type checking
test_is(is_listy(range(3)), True)
test_is(is_listy((i for i in range(3))), True)
test_is(is_listy(FC.L(1, 2, 3)), True)
test_is(is_listy({1, 2, 3}), True)
test_is(is_listy(r'raw string'), False)

test_is(is_listy_type(list), True)
test_is(is_listy_type(tuple), True)
test_is(is_listy_type(dict), True)
test_is(is_listy_type(str), False)
test_is(is_listy_type(NoneType), False)


## flatten

Recursively flatten nested collections into a single generator. Unlike `itertools.chain`, handles arbitrarily nested structures.

In [ ]:
def flatten(o, exclude=None):
    "Concatenate all collections and items as a generator"
    for item in o:
        if not is_listy(item, exclude): yield item; continue
        try: yield from flatten(item, exclude)
        except TypeError: yield item

In [ ]:
# Basic usage: flatten arbitrarily nested structures
test_eq(list(flatten([1, 2, 3])), [1, 2, 3])  # already flat
test_eq(list(flatten([1, [2, 3]])), [1, 2, 3])
test_eq(list(flatten([[1], [2], [3]])), [1, 2, 3])
test_eq(list(flatten((1, (2, 3)))), [1, 2, 3])  # works with tuples
test_eq(list(flatten([('a', (2,)), ('c', (4,))])), ['a', 2, 'c', 4])

In [ ]:
# Coverage: empty list
test_eq(list(flatten([])), [])


## Almost dataclass

A dataclass that allows you to specify fields with default values.

In [ ]:
def _flds(o, *args): return tuple(_ for _ in inspect.get_annotations(o).keys())+args

In [ ]:
class A:
    a: str
    b: str

_flds(A), _flds(A, 'c')

(('a', 'b'), ('a', 'b', 'c'))

In [ ]:
#| export
def _flds(o, *args): return tuple(_ for _ in inspect.signature(o if isinstance(o, type) else type(o)).parameters.keys())+args

In [ ]:
_flds(A)

()

In [ ]:
@FC.delegates(FC.store_attr, but=['names', 'self'])  # type: ignore
def Fields(*args, **kwargs):
    "Set annotated fields of `self` extracted from caller's locals; `*args` -> optional (None), `**kwargs` -> defaults"
    caller_locals =  sys._getframe(1).f_locals
    o = caller_locals['self']
    fields = {name: caller_locals[name] for name in _flds(o) if name in caller_locals}
    fields = {**fields, **{k: None for k in args}, **kwargs}
    # hack to make `store_attr` work with empty dict/list
    proxy = FC.AttrDict(_=None)  
    FC.store_attr(names=None, self=proxy, store_args=False, **fields)
    for k in fields: setattr(o, k, proxy[k])
    if not hasattr((cls := type(o)), '__fields__'): setattr(cls, '__fields__', tuple(fields.keys()))

In [ ]:
class A:
    a: str
    b: str
    def __init__(self, a:str,  b:str, c=None, d='ff'): Fields('e', f=7.3)#FC.store_attr(_flds(self), c=None, d=7.3)

o = A('aa', 'bb')
# test_eq(vars(o)['__stored_args__'], {'e': None, 'a': 'aa', 'b': 'bb', 'c': None, 'd': 'ff', 'f': 7.3})
test_eq(A.__fields__, ('a', 'b', 'c', 'd', 'e', 'f'))  # type: ignore
print(inspect.signature(A))
vars(o)

(a: str, b: str, c=None, d='ff')


{'a': 'aa', 'b': 'bb', 'c': None, 'd': 'ff', 'e': None, 'f': 7.3}

In [ ]:
class Chapter(FC.AttrDict):
    "A chapter of a book"
    def __init__(self, num:int, lines:slice, title:str, pov:str, content:str): Fields(ids={})
    @property
    def meta(self): return [self[_] for _ in type(self).__fields__ if _ not in ('ids', 'content')]  # type: ignore
    @property
    def text(self):
        r = self.lines
        return '/n'.join(lines[r.start:r.stop])  # type: ignore

_ch = Chapter(1, slice(0, 10), 'title', 'pov', 'contents')
_ch

```python
{ 'content': 'contents',
  'ids': {},
  'lines': slice(0, 10, None),
  'num': 1,
  'pov': 'pov',
  'title': 'title'}
```

In [ ]:
inspect.signature(Chapter)

<Signature (num: int, lines: slice, title: str, pov: str, content: str)>

In [ ]:
_ch.meta

[1, slice(0, 10, None), 'title', 'pov']

In [ ]:
class Book(list):
    "A book"
    title: str
    author: str
    def __init__(self, title,  author): Fields()
    @property
    def front_matter(self): return FC.AttrDict({k:getattr(self, k) for k in ('title', 'author')})
    @property
    def body(self) -> Sequence[Chapter]: return tuple(*self)

bk = Book('Tiempo Atrás', 'Rafael Marín')
bk, bk.front_matter, bk.body

([], {'title': 'Tiempo Atrás', 'author': 'Rafael Marín'}, ())

# String Utilities

## shorten

Truncate strings intelligently from left, right, or center. Useful for displaying long identifiers, paths, or data in logs and UIs.

In [ ]:
def shorten(x:Any, mode:Literal['l', 'r', 'c']='l', limit=40, trunc='…', empty='') -> str:
    "Truncate string `x` to `limit` chars from the left, right, or center, with a `trunc` marker."
    if len(s := str(x)) > limit:
        l, m, r = (
            (empty, trunc, s[-limit:]) if mode == 'l' else 
            (s[:limit], trunc, empty) if mode == 'r' else 
            (s[:(limit//2)-1], f" {trunc} ", s[-(limit//2-1):])
        )
        s = f'{l}{m}{r}'
    return s

def shortens(xs:Iterable[Any], mode:Literal['l', 'r', 'c']='l', limit=40, trunc='…', empty=''):
    "Yield truncated strings from `xs`, each shortened to `limit` chars."
    for x in xs: yield shorten(x, mode, limit, trunc, empty)

In [ ]:
# Basic usage: truncate from left (default), right, or center
uuid = 'ad2663b4-5ff5-40e3-a6ed-cc35f5627f8d'
test_eq(shorten(uuid, limit=17), '…a6ed-cc35f5627f8d')  # left truncation (default)
test_eq(shorten(uuid, 'r', limit=17), 'ad2663b4-5ff5-40e…')  # right truncation
test_eq(shorten(uuid, 'c', limit=17), 'ad2663b … 5627f8d')  # center truncation

test_eq(shorten('It was the best of times', limit=12), '…est of times')

In [ ]:
# Coverage: short strings (no truncation), numbers, empty
test_eq(shorten(0), '0')
test_eq(shorten(234), '234')
test_eq(shorten('asdfgh'), 'asdfgh')  # under limit, no change


In [ ]:
test_eq(list(shortens(['ad2663b4-5ff5-40e3-a6ed-cc35f5627f8d'], 'l', 10)), ['…35f5627f8d'])
test_eq(list(shortens(('abcdef', 'wertyu', 'sd'), 'r', 4)), ['abcd…', 'wert…', 'sd'])
test_eq(''.join(shortens('It was the best of times...'.split(), 'r', 1, '')), 'Iwtbot')

# Function Composition

## Runner

Create a function that runs multiple callables in sequence with the same arguments. Useful for side-effects like logging, validation, and updates without explicit chaining.

In [ ]:
_FuncItem: TypeAlias = Callable | Sequence['_FuncItem']

def Runner(*fns: _FuncItem) -> Callable:
    """Return a function that runs callables `fns` in sequence with same arguments. 
    Only side-effects, no composition."""
    _fns: tuple[Callable, ...] = tuple(FC.flatten(fns))  # type: ignore
    if not _fns: return FC.noop
    if len(_fns) == 1: return _fns[0]
    def _(*args, **kwargs) -> None:
        for f in _fns: f(*args, **kwargs)
    return _

In [ ]:
def f1(o,x,y=0): o.r += x+y
def f2(o,x,y=0): o.r += x*y
f3 = lambda o, x,y: print(o.r)

o = SimpleNamespace(r=0)
runner = Runner(f1, f2, f3)
runner(o, 2, 3)  # 11

Runner()(o, 2, 3)
test_eq(o.r, 11)
Runner([f1, f2], f3)(o, 2, 3)
test_eq(o.r, 22)
Runner([f1, [f2, f3]])(o, 2, 3)
test_eq(o.r, 33)

11


22


33


# Object Utilities

Helpers for manipulating objects and their attributes.

## setattrs

Copy attributes from one object/dict to another. Works bidirectionally: can copy from dict→object, object→dict, object→object, or dict→dict.

**Common use cases:**
- Initialize objects from configuration dicts
- Copy specific fields between instances
- Hydrate objects from API responses

**Parameters:**
- `dest`: Target object or dict to set attributes on
- `src`: Source object or dict to read from
- `flds`: Comma-separated field names (optional; defaults to all keys/public attrs)


In [ ]:
def setattrs(dest, src, flds=''):
    "Set `flds` or keys() or dir() attributes from `src` into `dest`"
    g = dict.get if isinstance(src, dict) else getattr
    s = operator.setitem if isinstance(dest, MutableMapping) else setattr
    if flds: flds = re.split(r",\s*", flds)
    elif isinstance(src, dict): flds = src.keys()
    else: flds = (_ for _ in dir(src) if _[0] != '_')
    for fld in flds: s(dest, fld, g(src, fld))

In [ ]:
class A: a = 1; b = 2; c = 3
a = A()
setattrs(a, {'b': 22, 'd': 44})
test_eq(a.a, 1)
test_eq(a.b, 22)
test_eq(a.c, 3)
test_eq(a.d, 44)  # type: ignore

## val_at - Implementation Evolution {.hidden}

The following cells document the development process of `val_at`, showing how the implementation evolved to handle increasingly complex cases (dicts → lists → mixed → JSON strings). This section is preserved for design rationale but can be skipped by most users.

**Skip to [Nested Data Access](#nested-data-access) if you just want to use the function.**

### Initial attempt: Simple dict-only version

In [ ]:
def val_at(element, json):
    return reduce(operator.getitem, element.split('.'), json)

j = {"app": {
    "Garden": {
        "Flowers": {
            "Red flower": "Rose",
            "White Flower": "Jasmine",
            "Yellow Flower": "Marigold"
        }
    },
    "Fruits": {
        "Yellow fruit": "Mango",
        "Green fruit": "Guava",
        "White Flower": "groovy"
    },
    "Trees": {
        "label": {
            "Yellow fruit": "Pumpkin",
            "White Flower": "Bogan"
        }
    }
}}

test_eq(val_at('app.Garden.Flowers.White Flower', j), 'Jasmine')

In [ ]:
print((dp := 'app.Garden.Flowers'), reduce(operator.getitem, dp.split('.'), j))

app.Garden.Flowers {'Red flower': 'Rose', 'White Flower': 'Jasmine', 'Yellow Flower': 'Marigold'}


In [ ]:
def val_at(element, j:str):
    d = json.loads(j)
    return reduce(lambda d, k: d[k] if isinstance(d, Mapping) else d[int(k)], element.split('.'), d)

j2 = {"app": {
    "Garden": {
        "Flowers": {
            "Red flower": "Rose",
            "White Flower": "Jasmine",
            "Yellow Flower": "Marigold"
        }
    },
    "Fruits": {
        "Yellow fruit": ["Mango", {"Banana": ["Canary Island", "Puerto Rico"]}],
        "Green fruit": "Guava",
        "White Flower": "groovy"
    },
    "Trees": {
        "label": {
            "Yellow fruit": "Pumpkin",
            "White Flower": "Bogan"
        }
    }
}}
test_eq(val_at('app.Fruits.Yellow fruit.1.Banana.0', json.dumps(j2)), 'Canary Island')

In [ ]:
apollo_astronauts = json.loads(Path('static/apollo_astronauts.json').read_text())
print((dp := 'Apollo 11.Michael Collins'), reduce(operator.getitem, dp.split('.'), apollo_astronauts))

Apollo 11.Michael Collins {'Experience': 'Pilot on Gemini 10 and Command Module pilot on Apollo 11.', 'Place in history': 'Collins was the first person to perform two EVAs in one mission.', 'Fast fact': 'Collins says his "secret terror" was returning to Earth alone if the surface mission failed.', 'Lunar wisdom': 'I really believe that if the political leaders of the world could see their planet from a distance of 100,000 miles their outlook could be fundamentally changed. That all-important border would be invisible, that noisy argument silenced.'}


In [ ]:
_T = TypeVar('_T')
_II = isinstance
def _at(d: Mapping|Sequence, k: str) -> Any: return (
    d[k] if _II(d, Mapping) else 
    d[int(k)] if _II(d, Sequence) and not _II(d, (str, bytes)) else 
    FC.stop(KeyError))  # type: ignore

def val_at(key_path: str, j: Mapping|Sequence|str|bytes|bytearray, default:_T=empty, sep:str='.') -> _T:
    "Return nested value at `key_path` from `j`. Raise if not found or `default` if not `empty`."
    try: return reduce(_at, key_path.split(sep), json.loads(j) if _II(j, (str, bytes, bytearray)) else j)
    except (KeyError, IndexError, ValueError) as e:
        if default is not empty: return default
        raise e

def key_at(key_path: str, j: Mapping|Sequence|str|bytes|bytearray, sep:str='.') -> bool:
    "Return `True` if nested `key_path` exists in `j`."
    try:
        reduce(_at, key_path.split(sep), json.loads(j) if _II(j, (str, bytes, bytearray)) else j)
        return True
    except (KeyError, IndexError):
        return False

In [ ]:
FC.test_fail(lambda: val_at(object, {}))  # type: ignore
FC.test_fail(lambda: val_at(object, []))  # type: ignore
FC.test_fail(lambda: val_at(object, object))  # type: ignore
FC.test_fail(lambda: val_at('', {}))
FC.test_fail(lambda: val_at('', []))
FC.test_fail(lambda: val_at('', object))   # type: ignore

FC.test_fail(lambda: val_at('a.b', {'a': 1}))
test_eq(val_at('a.b', {'a': 1}, None), None)

In [ ]:
d = [
    {'a': 1, 'b': [2,  3], 'c': {'d':   4}}, 
    {'a': 5, 'b': [6,  7],                 'd': [{'e': 81}, {'e': 82}]}, 
    {'a': 9, 'b': [10, 11], 'c': {'d': 12}}
]

test_fail(lambda: val_at('', d))
test_fail(lambda: val_at(object, d))  # type: ignore

test_fail(lambda: val_at('', object))  # type: ignore

test_eq(val_at('0.a', d), 1)
test_eq(val_at('1.b', d), [6, 7])
test_eq(val_at('2.c', d), {'d': 12})

In [ ]:
j2 = {
    "app": {
        "Garden": {
            "Flowers": {
                "Red flower": "Rose",
                "White Flower": "Jasmine",
                "Yellow Flower": "Marigold"
            }
        },
        "Fruits": {
            "Yellow fruit": ["Mango", {"Banana": ["Canary Island", "Puerto Rico"]}],
            "Green fruit": "Guava",
            "White Flower": "groovy"
        },
        "Trees": {
            "label": {
                "Yellow fruit": "Pumpkin",
                "White Flower": "Bogan"
            }
        },
        "Numbers": [1, 2, 3, 4, 5],
        "Boolean": True,
        "Null": None
    }
}

j2_str = json.dumps(j2)

test_eq(val_at('app.Fruits.Yellow fruit.1.Banana.0', j2_str), 'Canary Island')
test_eq(val_at('app.Garden.Flowers.Red flower', j2_str), 'Rose')
test_eq(val_at('app.Numbers.2', j2_str), 3)
test_eq(val_at('app.Boolean', j2_str), True)
test_eq(val_at('app.Null', j2_str), None)
test_fail(lambda: val_at('app.NonExistent', j2_str))
test_fail(lambda: val_at('app.Fruits.Yellow fruit.3', j2_str))
test_is(val_at('app.Fruits.Yellow fruit.3', j2_str, None), None)

test_eq(key_at('app.Garden.Flowers.Red flower', j2_str), True)
test_eq(key_at('app.Numbers.2', j2_str), True)
test_eq(key_at('app.Fruits.Yellow fruit.1.Banana.0', j2_str), True)
test_eq(key_at('app.NonExistent', j2_str), False)
test_eq(key_at('app.Fruits.Yellow fruit.3', j2_str), False)

## vals_at - Implementation Evolution {.hidden}

Similar to `val_at`, this section shows the iterative development of wildcard path traversal. The implementation needed to handle returning empty sentinels for missing paths while still extracting values from partial matches.

**Skip to [vals_at, vals_atpath](#vals_at-vals_atpath) for the final implementation and usage examples.**

### Early wildcard implementation

In [ ]:
_E = object()

def vals_at(path, d) -> empty | tuple[empty | object, ...] | object:
    "Return nested values-- or empty|(empty, ...)-- at `path` with wildcards '*' from `d`."
    curr, wc, rest = str(path).partition('*')
    if not wc and not rest:  return o if (o := val_at(curr, d, _E)) is not _E else empty
    o = val_at(curr.rstrip('.'), d, _E) if curr else d
    try: return (tuple(map(lambda x: empty if (res := vals_at(rest.lstrip('.'), x)) is _E else res, o))  # type: ignore
        if rest and o is not _E else o)
    except TypeError: return empty

test_eq(vals_at('a', object), empty)

In [ ]:
d = [
    {'a': 1, 'b': [2,  3],  'c': {'d':   4}}, 
    {'a': 5, 'b': [6,  7],                  'd': [{'e': 81}, {'e': 82}]}, 
    {'a': 9, 'b': [10, 11], 'c': {'d': 12}}
]

test_eq(vals_at('', d), empty)
test_eq(vals_at(object, d), empty)
test_eq(vals_at('*', d), (*d,))
test_eq(vals_at(2, d), d[2])
test_eq(vals_at('a', d), empty)
test_eq(vals_at('*.a', []), ())
test_eq(vals_at('*.a.*.b', [{'a': object}]), (empty,))

test_eq(vals_at('*,a', d), (empty, empty, empty))
test_eq(vals_at('*.a', d), (1, 5, 9))
test_eq(vals_at('*.a.*', d), (1,5,9))
test_eq(vals_at('*.b.1', d), (3, 7, 11))
test_eq(vals_at('*.c.d', d), (4, empty, 12))

test_eq(vals_at('*.d.*.e', d), (empty, (81, 82), empty))
test_eq(vals_at('*.d.*.f', d), (empty, (empty, empty), empty))
test_eq(vals_at('1.d.*.e', d), (81, 82))

In [ ]:
vals_at('*', d)

[{'a': 1, 'b': [2, 3], 'c': {'d': 4}},
 {'a': 5, 'b': [6, 7], 'd': [{'e': 81}, {'e': 82}]},
 {'a': 9, 'b': [10, 11], 'c': {'d': 12}}]

# Nested Data Access

Working with deeply nested data (JSON, API responses, config files) often requires verbose, error-prone access patterns. These utilities provide clean, safe access using dot notation or path sequences.

## val_at, val_atpath

Lookup values in nested structures using dot notation (`val_at`) or path sequences (`val_atpath`).

**Why use these?**
- Cleaner than chained `['key1']['key2'][0]` access
- Safe default values instead of try/except blocks
- Works with dicts, lists, objects, and JSON strings
- Handles mixed nesting (lists within dicts, etc.)

**Use `val_at`** when you have a dot-separated path string: `'user.addresses.0.city'`  
**Use `val_atpath`** when you have individual path components: `'user', 'addresses', 0, 'city'`

**Related utilities:**
- `has_key(o, 'dot.path')` / `has_path(o, *path)` - Check if path exists without retrieving value
- `vals_at(o, 'path.*.with.wildcards')` - Extract multiple values using wildcards (see below)

In [ ]:
_empty = Parameter.empty

In [ ]:
# def val_at(o, attr: str, default: Any=empty, sep='.'):
#     "Traverse nested `o` looking for attributes/items specified in dot-separated `attr`."
#     if not isinstance(attr, str): raise TypeError(f'{attr=!r} is not a string')
#     try:
#         for a in attr.split(sep):
#             if a[0]=='-' or a[0].isdigit(): a = int(a)
#             try: o = o[a]
#             except Exception:
#                 if isinstance(a, int):
#                     a = str(a)
#                     try: o = o[a]
#                     except Exception: pass
#                 o = getattr(o, a)
#     except Exception as e:
#         if default is not empty: return default
#         raise e
#     return o

def at_(
    o, # Object to traverse (dict, list, object, or nested combination)
    sym: str, # Path using dots and/or brackets (e.g., 'a.b[0].c' or 'a[b][c]')",
    default: Any=_empty, # Value to return if path not found (raises exception if not provided)
    sep='.' # Separator for path segments
) -> Any: # Value at the specified path
    "Traverse nested `o` using path `sym` with dot notation and/or bracket indexing"
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    try:
        for a in filter(None, sym.split(sep)):
            if a.lstrip('-').isdigit(): a = int(a)
            try: o = o[a]
            except Exception:
                if isinstance(a, int):
                    try: o = o[str(a)]; continue
                    except Exception: pass
                o = getattr(o, a)  # type: ignore
    except Exception:
        if default is not _empty: return default
        raise
    return o

val_at = at_

def val_atpath(o, *path: str|int,  default: Any=empty):
    "Traverse nested `o` looking for attributes/items specified in `path`."
    try:
        for a in path:
            try: o = o[a]
            except Exception: o = getattr(o, a)  # type: ignore
    except Exception:
        if default is not empty: return default
        raise
    return o

_NF = object()

def has_key(o, attr: str, sep='.') -> bool:
    "Return `True` if nested dot-separated `attr` exists."
    return val_at(o, attr, default=_NF, sep=sep) is not _NF

def has_path(o, *path: str|int) -> bool:
    "Return `True` if nested `path` exists."
    return val_atpath(o, *path, default=_NF) is not _NF

`at_` provides flexible path-based access to nested data structures:

**Supported types:** `o` can be/contains any combination of `Sequence`, `Mapping` (dicts, lists, tuples, L, etc), objects with \_\_getitem__, and/or dataclasses, objects with attributes accesible by `getattr`.

**Path syntax:**
- Dot notation: `'a.b.c'` accesses `o['a']['b']['c']` or `o.a.b.c`
- Bracket notation: `'a[b][c]'` is equivalent to `'a.b.c'`: `[x]` is a shorthand for `.x`
- Mixed: `'a.b[0].c'` combines both styles
- Numeric indices: `'items.2'` or `'items[2]'` for list/array access
- Empty path: `''` returns the object itself

**Access priority:** Item access (`[]`) is tried before attribute access (`.`)

**Error handling:** Raises exception if path not found, unless `default` is provided

In [ ]:
test_eq(at_({'a': 13}, 'a'), 13)

test_eq(at_({'a': {'b': 13}}, 'a.b'), 13)
test_eq(at_({'a': dict2obj({'b': 13})}, 'a.b'), 13)

test_eq(at_({'a': {'3': 7}}, 'a.3'), 7)

test_fail(lambda: at_(o, 'app.NonExistent'))
test_fail(lambda: at_(o, '[0'))
test_fail(lambda: at_(None, 'a'))
test_eq(at_(None, 'a', None), None)
test_fail(lambda: at_(None, 'a'))

test_eq(at_(o := {'meta': [1,2,3]}, ''), o)
test_eq(at_(o, 'meta[2]'), 3)
test_eq(at_(o, 'meta.2'), 3)

test_eq(at_('xyz', 'a', None), None)
test_eq(at_((s := 'xyz'), 'split'), s.split)

test_eq(at_([{'a':1}], '[0][a]'), 1)

In [ ]:
# Basic usage: access nested data with dot notation
data = {'user': {'name': 'Alice', 'scores': [10, 20, 30]}}
test_eq(val_at(data, 'user.name'), 'Alice')
test_eq(val_at(data, 'user.scores.1'), 20)  # list index

j2_list = [{'a':1}, {'b':2}]
test_eq(val_at(j2_list, '0'), {'a':1})
test_eq(val_at(j2_list, '0.a'), 1)

# Safe default instead of exception
test_eq(val_at(data, 'user.age', default=25), 25)

In [ ]:
test_eq(at_(j2_list, '0'), {'a':1})
test_eq(at_(j2_list, '1'), {'b':2})
test_eq(at_(j2_list, '[0][a]'), 1)
test_eq(at_(j2_list, '0.a'), 1)
test_eq(at_(j2_list, '[1].b'), 2)

In [ ]:
# Basic usage: access nested data with dot notation
data = {'user': {'name': 'Alice', 'scores': [10, 20, 30]}}
test_eq(val_at(data, 'user.name'), 'Alice')
test_eq(val_at(data, 'user.scores.1'), 20)  # list index

# Safe default instead of exception
test_eq(val_at(data, 'user.age', default=25), 25)

In [ ]:
# Works with lists of dicts (common in API responses)
records = [
    {'id': 1, 'items': [2, 3], 'meta': {'count': 4}}, 
    {'id': 5, 'items': [6, 7], 'nested': [{'val': 81}, {'val': 82}]}, 
    {'id': 9, 'items': [10, 11], 'meta': {'count': 12}}
]

test_eq(val_at(records, '0.id'), 1)
test_eq(val_at(records, '1.items'), [6, 7])
test_eq(val_at(records, '2.meta'), {'count': 12})

In [ ]:
test_eq(at_(records, '[2].meta'), {'count': 12})
test_eq(at_(records, '2[meta]'), {'count': 12})

In [ ]:
# Error handling: raises when path not found (unless default provided)
test_fail(lambda: val_at({}, 'a.b'))
test_fail(lambda: val_at([], 'a.b'))
test_fail(lambda: val_at({'a': 1}, 'a.b'))
test_fail(lambda: val_atpath({'a': 1}, 'a', 'b'))

# With default, no error
test_eq(val_at({'a': 1}, 'a.b', None), None)

In [ ]:
j2 = {
    "app": {
        "Garden": {
            "Flowers": {
                "Red flower": "Rose",
                "White Flower": "Jasmine",
                "Yellow Flower": "Marigold"
            }
        },
        "Fruits": {
            "Yellow fruit": ["Mango", {"Banana": ["Canary Island", "Puerto Rico"]}],
            "Green fruit": "Guava",
            "White Flower": "groovy"
        },
        "Trees": {
            "label": {
                "Yellow fruit": "Pumpkin",
                "White Flower": "Bogan"
            }
        },
        "Numbers": [1, 2, 3, 4, 5],
        "Boolean": True,
        "Null": None
    }
}

j2_str = j2#json.dumps(j2)

test_eq(val_at(j2_str, 'app.Fruits.Yellow fruit.1.Banana.0'), 'Canary Island')
test_eq(val_at(j2_str, 'app.Garden.Flowers.Red flower'), 'Rose')
test_eq(val_at(j2_str, 'app.Numbers.2'), 3)
test_eq(val_at(j2_str, 'app.Boolean'), True)
test_eq(val_at(j2_str, 'app.Null'), None)
test_fail(lambda: val_at(j2_str, 'app.NonExistent'))
test_fail(lambda: val_at(j2_str, 'app.Fruits.Yellow fruit.3'))
test_is(val_at(j2_str, 'app.Fruits.Yellow fruit.3', None), None)

In [ ]:
j2_obj = dict2obj(j2)

val_at(j2_obj, 'app.Fruits.Yellow fruit.1.Banana.0')
test_eq(val_at(j2_str, 'app.Null'), None)
test_eq(val_at(j2_str, 'app.Boolean'), True)
test_fail(lambda: val_at(j2_str, 'app.NonExistent'))

In [ ]:
test_eq(has_key(j2_str, 'app.Garden.Flowers.Red flower'), True)
test_eq(has_key(j2_str, 'app.Numbers.2'), True)
test_eq(has_key(j2_str, 'app.Fruits.Yellow fruit.1.Banana.0'), True)
test_eq(has_key(j2_str, 'app.NonExistent'), False)
test_eq(has_key(j2_str, 'app.Fruits.Yellow fruit.3'), False)

In [ ]:
test_eq(at_(j2, 'app[Numbers][2]'), 3)
test_eq(at_(j2, 'app[Fruits][Yellow fruit][1][Banana][0]'), 'Canary Island')
test_eq(at_(j2, 'app.Fruits[Yellow fruit].1.Banana[0]'), 'Canary Island')

test_eq(at_(dict2obj(j2), 'app[Fruits][Yellow fruit][1][Banana][0]'), 'Canary Island')
test_eq(at_(dict2obj(j2), 'app.Fruits[Yellow fruit].1.Banana[0]'), 'Canary Island')

test_is(at_(j2, '[bad][path]', 'default'), 'default')
test_is(at_(j2, '[missing]', None), None)
test_is(at_(j2, 'app[missing]', 'default'), 'default')
test_is(at_(j2, 'app[NonExistent]', None), None)

In [ ]:
test_fail(lambda:val_at(None, '0'))
test_eq(val_at(None, '0', None), None)
test_fail(lambda:val_atpath(None, 2))
test_eq(val_atpath(None, 2, default=None), None)
test_fail(lambda:val_atpath(None, -1))
test_eq(val_atpath(None, -1, default=None), None)
test_fail(lambda:val_at(None, '0.1'))
test_eq(val_at(None, '0.1', None), None)
test_fail(lambda:val_atpath(None, 4, 'b'))
test_eq(val_atpath(None, 4, 'b', default=None), None)

test_fail(lambda:val_atpath([], 1))
test_eq(val_atpath([], 1, default=None), None)
test_fail(lambda:val_atpath([], 'a', 2))
test_eq(val_atpath([], ('a', 2), default=None), None)  # type: ignore

test_fail(lambda:val_atpath({}, 1))
test_eq(val_atpath({}, 1, default=None), None)
test_fail(lambda:val_atpath(object(), '2'))
test_eq(val_at(object(), '0', None), None)
test_fail(lambda:val_at(object(), 'a.1'))
test_eq(val_at(object(), 'a.1', None), None)

In [ ]:
o = [1, 2, 3, [4, 5, 6], 7]
test_eq(val_at(o, '0'), 1)
test_eq(val_at(o, '-1'), 7)
test_eq(val_atpath(o, -1), 7)
test_eq(val_atpath(o, 2), 3)
test_eq(val_atpath(o, 3, 2), 6)
test_eq(val_atpath(o, 5, default=None), None)
test_eq(val_at(o, '3.4', None), None)

o = dict(a=1, b=2, c=3, d=dict(e=4, f=5), g=6)
test_eq(val_atpath(o, 'b'), 2)
test_eq(val_atpath(o, 'd', 'f'), 5)
test_eq(val_at(o, 'd.f'), 5)
test_eq(val_at(o, 'd.g', None), None)

(s := json.dumps(['foo', (1,2,3), {'bar': ('baz', None, 1.0, 2)}]))
o = json.loads(s)
test_eq(val_at(o, '0'), 'foo')
test_eq(val_atpath(o, 0), 'foo')
test_eq(val_at(o, '2.bar'), ['baz', None, 1.0, 2])
test_eq(val_atpath(o, 2, 'bar'), ['baz', None, 1.0, 2])
test_eq(val_at(o, '2.bar.3'), 2)
test_eq(val_at(o, '3.bar', None), None)
test_eq(val_atpath(o, 2, 'foo', default=None), None)

@dataclasses.dataclass
class _D:
    a: int
    b: str
    c: float
    d: dict[str, Any]
_d = _D(1, '2', 3.0, {'e': 4, 'f': '5', 'g': 6.0})

o = [0, 1, _d, 'a']
test_eq(val_atpath(o, 1), 1)
test_eq(val_at(o, '3'), 'a')
test_eq(val_atpath(o, 2), _d)
test_eq(val_at(o, '2.c'), 3.0)
test_eq(val_at(o, '2.d.f'), '5')

## vals_at, vals_atpath

Lookup **multiple** values using wildcards in paths. Think of it as a lightweight JSONPath for common cases.

**When to use:** Extract values from all items in a collection without explicit loops.

```python
data = [
    {'user': {'name': 'Alice', 'age': 30}},
    {'user': {'name': 'Bob', 'age': 25}},
]
vals_at(data, '*.user.name')  # ('Alice', 'Bob')
```

**Related utilities:**
- `val_at` / `val_atpath` - Single value extraction (see above)
- `filter_empty=True` parameter - Remove `empty` sentinels from results for cleaner output

In [ ]:
def _vals_atpath(o, *path: Any, filter_empty=False) -> empty | tuple[empty | object, ...] | object:
    try: 
        idx = path.index('*'); pre, pos = path[:idx], path[idx+1:]
    except ValueError:
        return a if (a := val_atpath(o, *path, default=_NF)) is not _NF else empty
    a = val_atpath(o, *pre, default=_NF) if pre else o
    if not pos: return a
    if a is _NF: return empty
    try: 
        res = tuple(map(lambda x: _vals_atpath(x, *pos, filter_empty=filter_empty), a))  # type: ignore
        if all(x is empty for x in res): return empty
        return tuple(filter(lambda x: x is not empty, res)) if filter_empty else res
    except (AttributeError, TypeError) as e: return empty

def vals_atpath(o, *path: Any, filter_empty=False) -> tuple[Any, ...]:
    "Return nested values-- or empty|(empty, ...)-- at `path` with wildcards '*' from `d`."
    if '*' not in path: return () if (res := val_atpath(o, *path, default=_NF)) is _NF else (res,)
    res = _vals_atpath(o, *path, filter_empty=filter_empty)
    return () if res is empty else res  # type: ignore

def _vals_at(o, path:str, filter_empty=False) -> empty | tuple[empty | object, ...] | object:
    pre, wc, pos = str(path).partition('*')
    if not wc and not pos: return a if (a := val_at(o, pre, _NF)) is not _NF else empty
    a = val_at(o, pre.rstrip('.'), _NF) if pre else o
    if not pos: return a
    if a is _NF: return empty
    try: 
        res = tuple(map(lambda x: _vals_at(x, pos.lstrip('.'), filter_empty=filter_empty), a))  # type: ignore
        if all(x is empty for x in res): return empty
        return tuple(filter(lambda x: x is not empty, res)) if filter_empty else res
    except TypeError: return empty

def vals_at(o, path:str, filter_empty=False) -> tuple[Any, ...]:
    "Return nested values-- or empty|(empty, ...)-- at `path` with wildcards '*' from `o`."
    if '*' not in path: return () if (res := val_at(o, path, _NF)) is _NF else (res,)
    res = _vals_at(o, path, filter_empty=filter_empty)
    return () if res is empty else res  # type: ignore


In [ ]:
test_eq(vals_atpath(object, ''), ())
test_eq(vals_atpath(object, 'a'), ())
test_eq(vals_atpath(object, '*', 'a'), ())
test_eq(vals_atpath(object, '*', 'a', '*'), ())

test_eq(vals_atpath(['a', 'b'], 'a'), ())
test_eq(vals_atpath(['a', 'b'], '*'), ('a', 'b'))
test_eq(vals_atpath(['a', 'b'], '*', 'a'), ())

test_eq(vals_atpath(['a', 'b'], 0), ('a',))
test_eq(vals_atpath(['a', 'b'], 2), ())

test_eq(vals_atpath([{'a':1}], 'a'), ())
test_eq(vals_atpath([{'a':1}], '*'), [{'a':1}])
test_eq(vals_atpath([{'a':1}, {'a':2}], '*', 'a'), (1, 2))

In [ ]:
test_eq(vals_at(object, ''), (object,))
test_eq(vals_at(object, 'a'), ())
test_eq(vals_at(object, '*.a'), ())
test_eq(vals_at(object, '*.a.*'), ())

test_eq(vals_at(['a', 'b'], 'a'), ())
test_eq(vals_at(['a', 'b'], '*'), ('a', 'b'))
test_eq(vals_at(['a', 'b'], '*.a'), ())

test_eq(vals_at(['a', 'b'], '0'), ('a',))
test_eq(vals_at(['a', 'b'], '2'), ())

test_eq(vals_at([{'a':1}], 'a'), ())
test_eq(vals_at([{'a':1}], '*'), [{'a':1}])
test_eq(vals_at([{'a':1}, {'a':2}], '*.a'), (1, 2))

In [ ]:
d = [
    {'a': 1, 'b': [2,  3],  'c': {'d':   4}}, 
    {'a': 5, 'b': [6,  7],                  'd': [{'e': 81}, {'e': 82}]}, 
    {'a': 9, 'b': [10, 11], 'c': {'d': 12}}
]

test_eq(vals_atpath(d, ''), ())
test_eq(vals_atpath(d, object), ())  # type: ignore
test_eq(vals_atpath(d, '*'), (*d,))
test_eq(vals_atpath(d, 2), (d[2],))
test_eq(vals_atpath(d, '2'), ())
test_eq(vals_atpath(d, 'a'), ())
test_eq(vals_atpath([], '*', 'a'), ())#())
test_eq(vals_atpath([{'a': object}], '*', 'a', '*', 'b'), ())

test_eq(vals_atpath(d, '*', 'a'), (1, 5, 9))
test_eq(vals_atpath(d, '*', 'a', '*'), (1,5,9))
test_eq(vals_atpath(d, '*', 'b', 1), (3, 7, 11))
test_eq(vals_atpath(d, '*', 'c', 'd'), (4, empty, 12))
test_eq(vals_atpath(d, '*', 'c', 'd', filter_empty=True), (4, 12))

test_eq(vals_atpath(d, '*', 'd', '*', 'e'), (empty, (81, 82), empty))
test_eq(vals_atpath(d, '*', 'd', '*', 'e', filter_empty=True), ((81, 82),))
test_eq(vals_atpath(d, '*', 'd', '*', 'f'), ())
test_eq(vals_atpath(d, 1, 'd', '*', 'e'), (81, 82))

In [ ]:
d = [
    {'a': 1, 'b': [2,  3],  'c': {'d':   4}}, 
    {'a': 5, 'b': [6,  7],                  'd': [{'e': 81}, {'e': 82}]}, 
    {'a': 9, 'b': [10, 11], 'c': {'d': 12}}
]

test_eq(vals_at(d, ''), (d,))
test_fail(lambda: vals_at(d, object))  # type: ignore
test_eq(vals_at(d, '*'), (*d,))
test_eq(vals_at(d, '2'), (d[2],))
test_eq(vals_at(d, 'a'), ())
test_eq(vals_at([], '*.a'), ())#())
test_eq(vals_at([{'a': object}], '*.a.*.b'), ())

test_eq(vals_at(d, '*,a'), ())
test_eq(vals_at(d, '*.a'), (1, 5, 9))
test_eq(vals_at(d, '*.a.*'), (1,5,9))
test_eq(vals_at(d, '*.b.1'), (3, 7, 11))
test_eq(vals_at(d, '*.c.d'), (4, empty, 12))
test_eq(vals_at(d, '*.c.d', filter_empty=True), (4, 12))

test_eq(vals_at(d, '*.d.*.e'), (empty, (81, 82), empty))
test_eq(vals_at(d, '*.d.*.e', filter_empty=True), ((81, 82),))
test_eq(vals_at(d, '*.d.*.f'), ())
test_eq(vals_at(d, '1.d.*.e'), (81, 82))

## deep_in

Check if a value exists anywhere in a nested structure (recursively searches dicts and iterables).

In [ ]:
def deep_in(o:Mapping|Iterable, val):
    "return True if val is in nested collections"
    if isinstance(o, Mapping):
        return val in o.values() or any(deep_in(v, val) for v in o.values() if isinstance(v, (Mapping, Iterable)))
    elif isinstance(o, Iterable):
        return val in o or any(deep_in(v, val) for v in o if isinstance(v, (Mapping, Iterable)))
    raise ValueError(f"deep_in: o must be a Mapping or Iterable, got {type(o)}")

In [ ]:
test_fail(lambda:deep_in(None, 1))  # type: ignore
test_eq(deep_in((), 1), False)
test_eq(deep_in((1,), 1), True)
test_eq(deep_in((1,), 2), False)
test_eq(deep_in({'a': 1}, 1), True)
test_eq(deep_in({'a': 1}, 2), False)
test_eq(deep_in({'a': (1,)}, 1), True)
test_eq(deep_in({'a': (1,)}, 2), False)
test_eq(deep_in({'a': {'b': 1}}, 1), True)
test_eq(deep_in({'a': {'b': 1}}, 2), False)
test_eq(deep_in({'a': {'b': (1,)}}, 1), True)
test_eq(deep_in({'a': {'b': (1,)}}, 2), False)

In [ ]:
def deep_in_v1(o:Mapping|Iterable, val):
    "return True if val is in o"
    if isinstance(o, Mapping):
        for v in o.values():
            if v == val or (is_listy(v) and deep_in_v1(v, val)): return True
        return False
    elif isinstance(o, Iterable):
        if val in o: return True
        return any(deep_in_v1(v, val) for v in o if is_listy(v))
    raise ValueError(f"deep_in: o must be a Mapping or Iterable, got {type(o)}")

In [ ]:
def deep_in_v2(o:Mapping|Iterable, val):
    stack = deque([o])
    while stack:
        curr = stack.popleft()
        if isinstance(curr, Mapping):
            if val in curr.values(): return True
            stack.extend(v for v in curr.values() if is_listy(v))
        elif isinstance(curr, Iterable):
            if val in curr: return True  
            stack.extend(v for v in curr if is_listy(v))
        else:
            raise ValueError(f"deep_in: o must be a Mapping or Iterable, got {type(curr)}")
    return False

In [ ]:
def deep_in_v3(o:Mapping|Iterable, val):
    def _search(obj):
        if isinstance(obj, Mapping):
            yield from obj.values()
            for v in obj.values():
                if is_listy(v): yield from _search(v)
        elif isinstance(obj, Iterable):
            yield from obj
            for v in obj:
                if is_listy(v): yield from _search(v)
        else:
            raise ValueError(f"deep_in: o must be a Mapping or Iterable, got {type(obj)}")
    
    return val in _search(o)

In [ ]:
test_data = [
    ((), 1), ((1,), 1), ((1,), 2), ({'a': 1}, 1), ({'a': 1}, 2),  
    ({'a': (1,)}, 1), ({'a': (1,)}, 2), ({'a': {'b': 1}}, 1), 
    ({'a': {'b': 1}}, 2), ({'a': {'b': (1,)}}, 1), ({'a': {'b': (1,)}}, 2)
]

for o, val in test_data:
    orig = deep_in(o, val)
    test_eq(deep_in_v1(o, val), orig)
    test_eq(deep_in_v2(o, val), orig) 
    test_eq(deep_in_v3(o, val), orig)

### Performance exploration

The following cells test alternative implementations of `deep_in` to find the best balance between clarity and speed. The chosen implementation (exported below) uses recursive traversal with early termination.


In [ ]:
deep_data = {'level1': {'level2': {'level3': {'level4': [1, 2, 3, {'deep': 'target'}]}}}}

def time_function(func, data, target, iterations=10000):
    start = time.time()
    for _ in range(iterations):
        func(data, target)
    return time.time() - start

print("\nPerformance comparison (10000 iterations):")
target = 'target'
times = {}
times['original'] = time_function(deep_in, deep_data, target)
times['v1_single_iter'] = time_function(deep_in_v1, deep_data, target)
times['v2_stack_based'] = time_function(deep_in_v2, deep_data, target)
times['v3_generator'] = time_function(deep_in_v3, deep_data, target)

for name, t in times.items():
    speedup = times['original'] / t if t > 0 else float('inf')
    print(f"{name:15}: {t:.4f}s (speedup: {speedup:.2f}x)")


Performance comparison (10000 iterations):


original       : 0.1300s (speedup: 1.00x)


v1_single_iter : 0.0907s (speedup: 1.43x)


v2_stack_based : 0.1032s (speedup: 1.26x)


v3_generator   : 0.1206s (speedup: 1.08x)


In [ ]:
def deep_in(o:Mapping|Iterable, val):
    "return True if val is in nested collections"
    if isinstance(o, Mapping):
        for v in o.values():
            if v == val or (is_listy(v) and deep_in(v, val)): return True
        return False
    elif isinstance(o, Iterable):
        if val in o: return True
        return any(deep_in(v, val) for v in o if is_listy(v))
    raise ValueError(f"deep_in: o must be a Mapping or Iterable, got {type(o)}")

## set_at

Set a value at a nested path in mappings, sequences, and objects. It follows the same dot/bracket path syntax as `val_at`, returning the assigned value.

In [ ]:
def set_at(o, sym:str, v, sep='.'):
    "Set existing value at nested path `sym`, using dot and bracket notation."
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    *pre,a = filter(None, sym.split(sep))
    o = at_(o, sep.join(pre), sep=sep)
    if a.lstrip('-').isdigit(): a = int(a)
    try: o[a]; o[a] = v
    except Exception:
        if isinstance(a, int):
            try: o[str(a)]; o[str(a)] = v; return v
            except Exception: pass
        getattr(o, a); setattr(o, a, v)
    return v

In [ ]:
data = dict(user=dict(scores=[10, 20, 30]))
set_at(data, 'user.scores[1]', 99), data

(99, {'user': {'scores': [10, 99, 30]}})

In [ ]:
data = dict(user=dict(name='Ada', scores=[10, 20, 30]), users=[dict(name='Bob')], **{'3': 'three'})
obj = SimpleNamespace(meta=SimpleNamespace(count=1))
ad = AD(a=AD(b=2))

test_eq(set_at(data, 'user.name', 'Grace'), 'Grace')
test_eq(data['user']['name'], 'Grace')

test_eq(set_at(data, 'user.scores.1', 99), 99)
test_eq(data['user']['scores'], [10, 99, 30])

test_eq(set_at(data, 'users[0].name', 'Lin'), 'Lin')
test_eq(data['users'][0]['name'], 'Lin')

test_eq(set_at(data, '3', 'numeric'), 'numeric')
test_eq(data['3'], 'numeric')

test_eq(set_at(obj, 'meta.count', 2), 2)
test_eq(obj.meta.count, 2)

test_eq(set_at(ad, 'a.b', 3), 3)
test_eq(ad.a.b, 3)

test_fail(lambda: set_at(data, 'user.age', 25))
test_fail(lambda: set_at(data, 'missing.name', 'x'))
test_fail(lambda: set_at(data, 'user.scores.9', 0))
test_fail(lambda: set_at((1, 2), '0', 3))
test_fail(lambda: set_at(data, '', None))

In [ ]:
def set_at(o, sym:str, v, create=False, sep='.'):
    "Set existing nested value at `sym`; with `create`, add only a final dict key or object attr."
    sym = re.sub(r'\[([^\]]+)\]', r'.\1', sym)
    *pre,a = filter(None, sym.split(sep))
    o = at_(o, sep.join(pre), sep=sep)
    a = int(a) if a.lstrip('-').isdigit() else a
    try: o[a]
    except Exception:
        if isinstance(a, int):
            try: o[str(a)]
            except Exception: pass
            else: o[str(a)] = v; return v
        if create and isinstance(o, MutableMapping): o[a] = v; return v
        if create and isinstance(a, str): setattr(o, a, v); return v
        getattr(o, a); setattr(o, a, v)
    else: o[a] = v
    return v

In [ ]:
data = dict(user=dict())
set_at(data, 'user.name', 'Ada', create=True), data

('Ada', {'user': {'name': 'Ada'}})

In [ ]:
data = dict(user=dict(name='Ada', scores=[10, 20, 30]), users=[dict(name='Bob')], **{'3': 'three'})
obj = SimpleNamespace(meta=SimpleNamespace(count=1))
ad = AD(a=AD(b=2))

test_eq(set_at(data, 'user.name', 'Grace'), 'Grace')
test_eq(data['user']['name'], 'Grace')
test_eq(set_at(data, 'user.scores.1', 99), 99)
test_eq(data['user']['scores'], [10, 99, 30])
test_eq(set_at(data, 'users[0].name', 'Lin'), 'Lin')
test_eq(data['users'][0]['name'], 'Lin')
test_eq(set_at(data, '3', 'numeric'), 'numeric')
test_eq(data['3'], 'numeric')
test_eq(set_at(obj, 'meta.count', 2), 2)
test_eq(obj.meta.count, 2)
test_eq(set_at(ad, 'a.b', 3), 3)
test_eq(ad.a.b, 3)

test_eq(set_at(data, 'user.age', 25, create=True), 25)
test_eq(data['user']['age'], 25)
test_eq(set_at(data, 'users[0].role', 'admin', create=True), 'admin')
test_eq(data['users'][0]['role'], 'admin')
test_eq(set_at(obj, 'meta.active', True, create=True), True)
test_eq(obj.meta.active, True)
test_eq(set_at(ad, 'a.c', 4, create=True), 4)
test_eq(ad.a.c, 4)

test_fail(lambda: set_at(data, 'user.city', 'London'))
test_fail(lambda: set_at(data, 'missing.name', 'x', create=True))
test_fail(lambda: set_at(data, 'user.missing.name', 'x', create=True))
test_fail(lambda: set_at(data, 'user.scores.3', 0, create=True))
test_fail(lambda: set_at(data, 'user.scores.9', 0, create=True))
test_fail(lambda: set_at((1, 2), '0', 3))
test_fail(lambda: set_at(data, '', None))

### Creation policy

`set_at` updates existing nested values by default. With `create=True`, it may create only the **final** mapping key or object attribute; every intermediate path component must already exist.

Mutable sequences are deliberately conservative: existing integer indices may be replaced, but `create=True` never appends, extends, or fills a list. An index equal to `len(seq)` and all out-of-range indices raise. This avoids silently choosing append semantics or placeholder values.

If sequence creation is needed later, add it as an explicit, separate option rather than broadening `create=True`.

### Numeric mapping keys

Path components that look like integers first use integer indexing, then fall back to an existing string key. With `create=True`, a missing numeric-looking final component currently creates an integer mapping key—for example, `set_at({}, '3', x, create=True)` creates `{3: x}`. Preserve or revise this deliberately if string-key creation is preferred.

# Dictionary Helpers

Utilities for common dictionary operations: bulk popping, getting multiple keys, conditional updates.

## pops_

Pop multiple keys from a dict into a new dict. Useful for extracting specific kwargs or splitting a config dict.

**Related:** `pops_values_` returns a tuple of values instead of a dict; `gets` for non-destructive retrieval.

In [ ]:
def pops_(d: dict, *ks: Hashable) -> dict:
    "Pop existing `ks` items from `d` in-place into a dictionary."
    return {k:d.pop(k) for k in ks if k in d}

In [ ]:
# Usage: extract specific keys from a dict
d = {'a': 1, 'b': 2, 'c': 3}
extracted = pops_(d, 'a', 'b')
test_eq(extracted, {'a': 1, 'b': 2})
test_eq(d, {'c': 3})  # original dict modified

In [ ]:
# Coverage: missing keys, empty dict, duplicates
test_eq(pops_({'a': 1, 'b': 2, 'c': 3}, 'd'), {})  # missing keys ignored
test_eq(pops_({'a': 1, 'b': 2, 'c': 3}, 'a', 'c', 'd'), {'a': 1, 'c': 3})  # some missing
test_eq(pops_({}, 'a'), {})  # empty dict
test_eq(pops_({'a': 1}, 'a', 'a'), {'a': 1})  # duplicate keys


## pops_values_

Like `pops_` but returns a tuple of values instead of a dict. Missing keys get `Parameter.empty` as placeholder. Useful for unpacking: `a, b, c = pops_values_(d, 'a', 'b', 'c')`

**Related:** `pops_` returns a dict; `gets` for non-destructive retrieval.

In [ ]:
def pops_values_(d: dict, *ks: Hashable) -> tuple:
    "Pop existing `ks` items from `d` in-place into a tuple of values or `Parameter.empty` for missing keys."
    return tuple(d.pop(k, Parameter.empty) for k in ks)

In [ ]:
test_eq(pops_values_({'a': 1, 'b': 2, 'c': 3}, 'a', 'b'), (1, 2))
test_eq(pops_values_({'a': 1, 'b': 2, 'c': 3}, 'd'), (Parameter.empty,))
test_eq(pops_values_({'a': 1, 'b': 2, 'c': 3}, 'a', 'c', 'd'), (1, 3, Parameter.empty))
test_eq(pops_values_({}, 'a'), (Parameter.empty,))
test_eq(pops_values_({'a': 1}, 'a', 'a'), (1, Parameter.empty))

## gets

Fetch multiple values from a dict as a tuple. Missing keys return `Parameter.empty`. Useful for unpacking multiple config values at once.

**Related:** `pops_` / `pops_values_` for destructive (modifying) retrieval.

In [ ]:
def gets(d: Mapping, *ks: Hashable):
    "Fetches `ks` values, or `Parameter.empty` for missing keys, from `d` into a tuple."
    return tuple(d.get(k, Parameter.empty) for k in ks)  # type: ignore

In [ ]:
# Usage: unpack multiple config values at once
config = {'host': 'localhost', 'port': 8080, 'debug': True}
host, port = gets(config, 'host', 'port')
test_eq((host, port), ('localhost', 8080))

# Missing keys return Parameter.empty
test_eq(gets({'a': 1, 'b': 2}, 'a', 'c', 'b'), (1, Parameter.empty, 2))

In [ ]:
# Coverage: empty args, Path keys, reordering
test_eq(gets({'a': 1, 'b': 2}), ())  # no keys requested
a, b = gets({'a': 1, 'b': 2}, 'b', 'a')  # reordered
test_eq((a, b), (2, 1))

d = {Path('a'): 1, Path('b'): 2}  # non-string keys
test_eq(gets(d, Path('a'), Path('c'), Path('b')), (1, Parameter.empty, 2))


## update_

Update dict/object with kwargs, skipping values equal to `empty_value`. Useful for conditional updates where you don't want to set `None` or other sentinel values.

In [ ]:
# def update_(d:dict|None=None, /, empty_value=None, **kwargs):
#     "Update `d` in-place with `kwargs` whose values aren't `empty_value`"
#     d = d if d is not None else {}
#     for k, v in kwargs.items():
#         if v is not empty_value: d[k] = v
#     return d

In [ ]:
def update_(dest=None, /, empty_value=None, **kwargs) -> Any:
    "Update `dest` in-place with `kwargs` whose values aren't `empty_value`"
    dest = dest if dest is not None else {}
    f = operator.setitem if isinstance(dest, MutableMapping) else setattr
    for k, v in filter(lambda x: x[1]!=empty_value, kwargs.items()): f(dest, k, v)
    return dest

Helper for conditionally updating dictionaries/namespaces.


In [ ]:
# Usage: conditionally update dict (skip None by default)
d = {'a': 1}
update_(d, b=2, c=None)
test_eq(d, {'a': 1, 'b': 2})  # c=None was skipped

# Custom empty_value sentinel
d = {}
update_(d, a=True, b=Parameter.empty, empty_value=Parameter.empty)
test_eq(d, {'a': True})  # b was skipped

In [ ]:
# Works with objects too (uses setattr)
class Config: host = 'localhost'; port = 8000
cfg = update_(Config(), host='0.0.0.0', timeout=None)
test_eq(cfg.host, '0.0.0.0')
test_eq(cfg.port, 8000)  # unchanged
test_is(hasattr(cfg, 'timeout'), False)  # None was skipped

In [ ]:
# Coverage: create new dict, AD dict
d = update_(a=1, b=None)  # dest=None creates new dict
test_eq(d, {'a': 1})

ad = AD(a=1, b=2)
update_(ad, a=3, b=4, c=5)
test_eq(ad.a, 3)
test_eq(ad.c, 5)


In [ ]:
ad = AD(a=1, b=2)
update_(ad, a=3, b=4, c=5)
test_eq(ad.a, 3)
test_eq(ad.b, 4)
test_eq(ad.c, 5)

# _get_globals

Internal utility for accessing the caller's global namespace. This enables functions to dynamically resolve names from the calling module's context, useful for metaprogramming and dynamic imports.

**Note:** This is an internal helper (marked `#| exporti`). Most users won't need it directly.

In [ ]:
#| exporti

def _get_globals(mod: str):
    if hasattr(sys, '_getframe'):
        glb = sys._getframe(2).f_globals
    else:
        glb = sys.modules[mod].__dict__
    return glb

In [ ]:
def _gtest(): return _get_globals(__name__)
g1 = _gtest()
g2 = globals()
test_eq(g1, g2)

# Path Utilities

## bundle path
Get the directory containing a module. Useful for finding resources bundled with your package.

In [ ]:
def bundle_path(mod:str|ModuleType):
    "Return the path to the module's directory or current directory."
    if isinstance(mod, str): mod = importlib.import_module(mod)
    return Path(fn).parent if (fn := getattr(mod, '__file__', None)) else Path()

In [ ]:
import pote

In [ ]:
test_eq(bundle_path(__name__), Path('.'))
test_eq(bundle_path('pote').resolve(), Path(pote.__file__).parent)

## Usable dir
Utilities for selecting an app-data directory that is likely to work across normal systems and restricted sandboxed environments.

In [ ]:
def usable_dir(p:Path) -> bool:
    "Check whether `dir` is likely usable for local app data."
    try: p.mkdir(parents=True, exist_ok=True)
    except Exception: return False
    return p.is_dir() and os.access(p, os.R_OK|os.W_OK|os.X_OK)

def first_usable_dir(*ps) -> Path:
    'Return first path in `ps` that is likely a usable dir in sandboxed envs.'
    for p in ps:
        try: p = Path(p).expanduser().resolve()
        except Exception: continue
        if usable_dir(p): return p
    raise FileNotFoundError("No usable directory found")

In [ ]:
cwd = Path.cwd()
base_pth = first_usable_dir(xdg_data_home()/'myapp', (cwd.parent if cwd.name == 'nbs' else cwd)/'.myapp')
backups_dir = base_pth/'backups'


Choose the first "usable" directory from a list of candidates, preferring standard platform locations while allowing project-local fallbacks for agents or sandboxes with limited filesystem access. Intended for lightweight app data such as caches, backups, and generated files where the exact location matters less than having a reliable place to read and write.

# ID Generation

Utilities for generating unique identifiers in the current session.

## Kounter

A callable counter that increments and returns the count for each key. Useful for generating sequential IDs grouped by type.

In [ ]:
class Kounter(defaultdict):
    def __init__(self): super().__init__(int)
    def __call__(self, k): self[k] += 1; return self[k]

In [ ]:
cntr = Kounter()
cntr('a'), cntr('b'), cntr('a'), cntr('a'), cntr('b'), cntr('b'), cntr('b')
test_eq(dict(cntr), {'a': 3, 'b': 4})
test_eq(cntr('int'), 1)

## id_gen

Create a function that generates unique IDs for objects. Without arguments returns a random ID, with an object returns `TypeName_N` where N increments per type.

In [ ]:
lines = Path("static/wordlist.txt").read_text().splitlines()
words = [line.strip() for line in lines if line.isalpha()]

In [ ]:
def modify_word(word):
    # Randomly capitalize the first or second letter
    if len(word) > 1:
        idx_to_capitalize = random.choice([0, 1])
        word = word[:idx_to_capitalize] + word[idx_to_capitalize].upper() + word[idx_to_capitalize + 1:]
    else:
        word = word.upper()  # If single letter, capitalize it
    
    # Randomly add a number (0–99) at the start or end
    if random.choice([True, False]):
        number = random.randint(0, 99)
        # if random.choice([True, False]):
        #     word = f"{number}{word}"  # Number at the start
        # else:
        word = f"{word}{number}"  # Number at the end
    
    return word

def generate_readable_id(num_words=3):
    words_part = [modify_word(random.choice(words)) for _ in range(num_words)]
    id_candidate = '-'.join(words_part)

    # Ensure it's a valid CSS identifier
    if not re.match(r"^[a-zA-Z_][\w\-]*$", id_candidate):  # Add '_' if invalid
        id_candidate = f"_{id_candidate}"
    
    return f"{id_candidate}-{random.randint(0, 9999)}"

In [ ]:
generate_readable_id(), generate_readable_id()

('Operating-pRocessed90-Operating71-2758', 'Park-Primary68-Editor-4393')

In [ ]:
def long_id(pre='b'):
    return pre+hexlify(os.urandom(16), '-', 4).decode('ascii')

def simple_id(n=4, pre=''): return pre+hexlify(os.urandom(n)).decode()

def id_gen():
    kntr = Kounter()
    def _(o:Any=None, pre=''): 
        if o is None: return simple_id()
        o = o if isinstance(o, str) else type(o).__name__
        return f"{pre}{o}_{kntr(o)}"
    return _

The `id_gen` factory creates a function that takes any object and generates an unique Id valid during the current session. Useful for creating unique element IDs in dynamic HTML content.


In [ ]:
new_id = id_gen()
new_id(), new_id()

('b2d165c8', 'fc448215')

In [ ]:
int_id = id_gen()
int_id(7), int_id(8)

('int_1', 'int_2')

In [ ]:
obj_id = id_gen()
o1, o2 = object(), object()
test_eq((obj_id(o1), obj_id(o2)), ('object_1', 'object_2'))

hi_id = id_gen()
test_eq((hi_id('hello'), hi_id('bye', '-'), hi_id('hello')), ('hello_1', '-bye_1', 'hello_2'))

dict_id = id_gen()
test_eq((dict_id(d1 := {'a': 1}), dict_id(d2 := {'a': 1})), ('dict_1', 'dict_2'))

pth_id = id_gen()
test_eq((pth_id(Path('.')), pth_id(Path()), pth_id(Path('./bin'))), ('PosixPath_1', 'PosixPath_2', 'PosixPath_3'))

# Metaclasses

## WithCounterMeta

A metaclass that adds automatic instance counting. Each instance gets a `_cnt_` attribute with its creation order number.

In [ ]:
class WithCounterMeta(FC.FixSigMeta):
    "Adds a `_cnt_` attribute to its classes and increments it for each new instance."
    _cnt_: int
    def __call__(cls, *args, **kwargs):
        res = super().__call__(*args, **kwargs)
        res._cnt_ = cls._cnt_
        cls._cnt_ += 1
        return res
    def __new__(cls, name, bases, dict):
        res = super().__new__(cls, name, bases, dict)
        res._cnt_ = 0
        return res

In [ ]:
class _T1(metaclass=WithCounterMeta): pass
class _T2(metaclass=WithCounterMeta): pass

test_is(_T1._cnt_, 0)
test_is(_T2._cnt_, 0)

test_is(_T1()._cnt_, 0)
test_is(_T1._cnt_, 1)
test_is(_T2._cnt_, 0)

test_is(_T1()._cnt_, 1)
test_is(_T2()._cnt_, 0)
test_eq(_T1._cnt_, 2)
test_eq(_T2._cnt_, 1)

# export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nb_path = '00_basic.ipynb'
nbdev_export(nb_path)